# Правила мониторинга ПКБ: какие влияют на переход в DPD 31+

**Что делает ноутбук.** Для каждого правила мониторинга ПКБ отвечает: повышает оно риск, снижает или не влияет.

1. **Период.** Берутся только эпизоды, вышедшие в просрочку (DPD 1) за то время, когда есть данные мониторинга ПКБ (`hes`). У каждого эпизода окно наблюдения целиком лежит внутри периода ПКБ. Если в ПКБ есть дни без данных (провал загрузки), эпизоды, чьё окно попадает на эти дни, исключаются.
2. **«Плохой» эпизод** — дошёл до DPD 31+.
3. **Событие правила засчитывается**, если оно пришло в банк (`created_date`) за `WINDOW_DAYS` дней до выхода в просрочку. События по нашим собственным договорам не учитываются: они повторяют то, что банк и так знает.
4. **Направление изменения берётся из JSON.** Если в `old_value` и `new_value` есть одно и то же числовое поле (долг, сумма просрочки, платёж, дни просрочки), правило делится на «рост» и «снижение». Статусы делятся на «установлен» и «снят». Например, «Совокупная сумма основного долга увеличилась/уменьшилась» превращается в две строки: «— рост» и «— снижение».
5. **По каждой строке считается**, во сколько раз доля «плохих» у эпизодов с событием выше или ниже, чем без события, с 95% доверительным интервалом (ДИ), и выводится вывод.

**Как запустить.** В памяти должны быть таблицы `ovrd_contr` и `hes` (как для большого ноутбука). Можно вместо этого указать пути к файлам в настройках ниже. Затем запустите все ячейки (Run All).

**Результат:** короткий итог, одна таблица «Итог по правилам ПКБ», один график и файл Excel.

In [ ]:
# ---------------- НАСТРОЙКИ ----------------
OVRD_PATH = None      # путь к файлу ovrd_contr (.parquet / .csv / .pkl), если таблица ovrd_contr не загружена в память
HES_PATH = None       # путь к файлу hes, если таблица hes не загружена в память

WINDOW_DAYS = 14      # событие засчитывается, если пришло за столько дней до выхода в просрочку (можно 7 или 30)
BAD_DPD = 31          # «плохой» эпизод: дошёл до этого DPD
MIN_EPISODES = 200    # меньше эпизодов с событием — «мало данных»
MIN_BADS = 30         # меньше «плохих» с событием (наблюдаемых и ожидаемых) — «мало данных»
EXCLUDE_GAP_EPISODES = True   # исключать эпизоды, у которых окно попадает на дни без данных ПКБ
OUT_XLSX = 'pkb_rules_summary.xlsx'

# имена колонок
C_CONTRACT, C_PROCESS, C_IIN, C_ACT, C_DPD = 'contract_number', 'process_ref', 'iin', 'act_date', 'ovd_days_cnt'
H_IIN, H_RULE, H_OLD, H_NEW, H_TIME = 'bin_iin', 'rule_name', 'old_value', 'new_value', 'created_date'

import json, math, re, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import orjson                      # если установлен, разбор JSON быстрее
    _loads = orjson.loads
except ImportError:
    _loads = json.loads
try:
    display                            # noqa: F821
except NameError:
    display = print


In [ ]:
# ---------------- ЗАГРУЗКА И ОЧИСТКА КЛЮЧЕЙ ----------------
def load(name, path, cols):
    """Таблица из памяти (если path = None) или из файла — только нужные колонки."""
    if path is None:
        if name in globals():
            return globals()[name]
        raise ValueError(f'Нет таблицы {name}: загрузите её в память или укажите путь в настройках')
    p = str(path).lower()
    if p.endswith('.parquet'):
        return pd.read_parquet(path, columns=cols)
    if p.endswith('.pkl'):
        return pd.read_pickle(path)
    return pd.read_csv(path, usecols=lambda c: c in cols)


def to_dt(s):
    """Дата/время без часового пояса (время Казахстана, UTC+5)."""
    s = pd.Series(s, copy=False)
    if not pd.api.types.is_datetime64_any_dtype(s):
        try:
            s = pd.to_datetime(s, errors='coerce', format='ISO8601')
        except (TypeError, ValueError):
            s = pd.to_datetime(s, errors='coerce')
    if getattr(s.dt, 'tz', None) is not None:
        s = s.dt.tz_convert('Etc/GMT-5').dt.tz_localize(None)
    return s


def norm_iin(s):
    """ИИН: 12 цифр с ведущими нулями."""
    s = pd.Series(s, copy=False).astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    return s.where(s.str.fullmatch(r'\d{1,12}')).str.zfill(12)


def num(x):
    """Целое число с пробелами между разрядами: 108 000."""
    return f'{int(x):,}'.replace(',', ' ')


def norm_code(s):
    """Номер договора: без пробелов по краям, в верхнем регистре; пустые значения -> NaN."""
    s = pd.Series(s, copy=False).astype(str).str.strip().str.upper()
    return s.where(~s.isin(['', 'NAN', 'NONE', 'NULL', '<NA>', 'NAT']))


ovrd = load('ovrd_contr', OVRD_PATH, [C_CONTRACT, C_PROCESS, C_IIN, C_ACT, C_DPD])
hes_src = load('hes', HES_PATH, [H_IIN, H_RULE, H_OLD, H_NEW, H_TIME])
print(f'ovrd_contr: {num(len(ovrd))} строк; hes: {num(len(hes_src))} строк')


In [ ]:
# ---------------- ПЕРИОД ДАННЫХ ПКБ И ЭПИЗОДЫ ПРОСРОЧКИ ----------------
fmt = lambda d: pd.Timestamp(d).strftime('%d.%m.%Y')                         # noqa: E731

# 1) период мониторинга ПКБ по дням поступления событий; дни с объёмом меньше 20% от обычного для этого дня недели —
#    дни без данных (провал загрузки)
h_time = to_dt(hes_src[H_TIME])
daily = h_time.dt.normalize().value_counts().sort_index()
normal = daily[daily >= 0.2 * daily.groupby(daily.index.dayofweek).transform('median')]
HES_START, HES_END = normal.index.min(), normal.index.max()
all_days = pd.date_range(HES_START, HES_END, freq='D')
full = daily.reindex(all_days).fillna(0)
gap_days = all_days[(full < 0.2 * full.groupby(full.index.dayofweek).transform('median')).values]

# 2) эпизоды просрочки: дата выхода в просрочку = дата среза − (DPD − 1)
d = pd.DataFrame({'contract': norm_code(ovrd[C_CONTRACT]).values, 'iin': norm_iin(ovrd[C_IIN]).values,
                  'act_date': to_dt(ovrd[C_ACT]).dt.normalize().values,
                  'dpd': pd.to_numeric(ovrd[C_DPD], errors='coerce').values})
d = d.dropna(subset=['contract', 'act_date', 'dpd'])
d = d[d['dpd'] >= 1]
DATA_END = d['act_date'].max()
n_dates = d.loc[d['act_date'] > DATA_END - pd.Timedelta(days=60), 'act_date'].nunique()   # срезы ежедневные?
d = d.groupby(['contract', 'act_date'], sort=False).agg(dpd=('dpd', 'max'), iin=('iin', 'first')).reset_index()
d['dlq_start'] = d['act_date'] - pd.to_timedelta(d['dpd'] - 1, unit='D')
d = d.sort_values(['contract', 'dlq_start', 'act_date'], kind='mergesort')
new_ep = d.groupby('contract')['dlq_start'].diff().dt.days.fillna(99) > 1          # разрыв больше дня — новый эпизод
d['ep'] = new_ep.astype(int).groupby(d['contract']).cumsum()
ep = (d.groupby(['contract', 'ep'], sort=False)
        .agg(iin=('iin', 'first'), dlq_start=('dlq_start', 'min'), max_dpd=('dpd', 'max'), last_seen=('act_date', 'max'))
        .reset_index().sort_values(['contract', 'dlq_start'], kind='mergesort'))
# новая дата начала внутри ещё открытого эпизода (после частичной оплаты) — не новый выход в просрочку
ep = ep[~(ep['dlq_start'] <= ep.groupby('contract')['last_seen'].shift())]

# 3) популяция: окно наблюдения целиком внутри периода ПКБ, исход (DPD 31+) уже можно увидеть
P_FROM = HES_START + pd.Timedelta(days=WINDOW_DAYS)
P_TO = min(HES_END, DATA_END - pd.Timedelta(days=BAD_DPD - 1))
pop = ep[(ep['dlq_start'] >= P_FROM) & (ep['dlq_start'] <= P_TO) & ep['iin'].notna()].copy()
n_before_gap = len(pop)
if EXCLUDE_GAP_EPISODES and len(gap_days):
    hit = np.zeros(len(pop), dtype=bool)
    for g in gap_days:                                   # окно эпизода: [дата выхода − WINDOW_DAYS, дата выхода)
        hit |= ((pop['dlq_start'] - pd.Timedelta(days=WINDOW_DAYS) <= g) & (g < pop['dlq_start'])).values
    pop = pop[~hit]
pop['bad'] = (pop['max_dpd'] >= BAD_DPD).astype(int)
pop = pop.reset_index(drop=True)
pop['ep_id'] = np.arange(len(pop))
N, B = len(pop), int(pop['bad'].sum())


def gap_text(days_):
    """Дни одной строкой, подряд идущие — диапазоном: 07.10.2026–20.10.2026, ..."""
    if not len(days_):
        return 'нет'
    s, out = pd.Series(days_), []
    grp = (s.diff().dt.days != 1).cumsum()
    for _, g in s.groupby(grp):
        out.append(fmt(g.iloc[0]) if len(g) == 1 else f'{fmt(g.iloc[0])}–{fmt(g.iloc[-1])}')
    return ', '.join(out)


A_TEXT = gap_text(pd.DatetimeIndex(sorted(pop['dlq_start'].unique()))) if N else 'нет'   # фактические даты выхода


print(f'Данные мониторинга ПКБ: с {fmt(HES_START)} по {fmt(HES_END)}; дни без данных: {gap_text(gap_days)}.')
print(f'Данные о просрочке (ovrd_contr) — по {fmt(DATA_END)}.')
print(f'Анализ: эпизоды с выходом в просрочку с {fmt(P_FROM)} по {fmt(P_TO)}; события ПКБ — за {WINDOW_DAYS} дн. до выхода.')
if EXCLUDE_GAP_EPISODES and len(gap_days):
    print(f'Исключено эпизодов, у которых окно попадает на дни без данных ПКБ: {num(n_before_gap - N)}. '
          f'Фактически в анализе эпизоды с выходом в просрочку: {A_TEXT}.')
print(f'Эпизодов в анализе: {num(N)}; «плохих» (DPD {BAD_DPD}+): {num(B)} ({B / max(N, 1):.1%}).')
if n_dates < 45:
    print(f'ВНИМАНИЕ: в ovrd_contr не ежедневные срезы (за последние 60 дней только {n_dates} дат среза). Короткие '
          'эпизоды не видны, а выход на DPD 31+ между срезами пропускается — доля «плохих» и выводы будут искажены.')
if N == 0:
    raise ValueError('В анализ не попало ни одного эпизода: проверьте даты в ovrd_contr и hes, уменьшите WINDOW_DAYS '
                     'или поставьте EXCLUDE_GAP_EPISODES = False.')
if N < 5000:
    print('ВНИМАНИЕ: эпизодов мало — выводы будут неточными. Можно уменьшить WINDOW_DAYS '
          'или поставить EXCLUDE_GAP_EPISODES = False.')


In [ ]:
# ---------------- СОБЫТИЯ ПКБ: ОТБОР И РАЗБОР JSON ----------------
# 1) только клиенты из популяции и только события, которые могут попасть в окно
iins = pd.unique(pop['iin'])
raw = hes_src[H_IIN]
m = ((h_time >= P_FROM - pd.Timedelta(days=WINDOW_DAYS)) & (h_time < P_TO)).values
if pd.api.types.is_numeric_dtype(raw):
    m = m & raw.isin({int(x) for x in iins}).values
else:
    hit = raw.isin(set(iins) | {x.lstrip('0') for x in iins}).values    # быстрый вариант: ИИН записаны как есть
    if not hit.any():                                                    # ИИН с пробелами / '.0': медленная очистка
        hit = norm_iin(raw).isin(set(iins)).values
    m = m & hit
ev = pd.DataFrame({'iin': norm_iin(raw[m]).values, 'rule': hes_src.loc[m, H_RULE].astype(str).str.strip().values,
                   'time': h_time[m].values, 'old': hes_src.loc[m, H_OLD].values, 'new': hes_src.loc[m, H_NEW].values})
ev = ev[ev['rule'].ne('') & ~ev['rule'].str.lower().isin(['nan', 'none', 'null'])]
print(f'Событий ПКБ по клиентам популяции в нужные даты: {num(len(ev))}')

# 2) JSON: номер договора и направление изменения
# поле JSON, о котором говорит название правила (если в JSON несколько числовых полей, сравнивается это)
RULE_FIELD = [('основного долга', 'debt_sum'), ('дней просрочки', 'overdue_instalment_count'),
              ('сумма просрочки', 'overdue_sum'), ('ежемесячного платежа', 'instalment')]
MEASURES = ['overdue_instalment_count', 'overdue_sum', 'debt_sum', 'instalment', 'outstanding_debt', 'zadolzhenost']
NOT_MEASURES = ('id', 'contract_code', 'credit_app_id', 'credit_report_id', 'group_id', 'internal_event_id')


def as_dict(x):
    if isinstance(x, dict):
        return x
    if not isinstance(x, (str, bytes)) or len(x) < 2:
        return None
    try:
        v = _loads(x)
    except Exception:
        return None
    return v if isinstance(v, dict) else None


def as_num(v):
    if isinstance(v, bool):
        return float(v)
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        t = v.strip().lower()
        if t in ('true', 'false'):
            return float(t == 'true')
        t = re.sub(r"[\s']", '', t)                         # пробелы, в т.ч. неразрывные
        if ',' in t and '.' in t:                           # 1,234,567.89 или 1.234.567,89
            t = t.replace(',', '') if t.rfind('.') > t.rfind(',') else t.replace('.', '').replace(',', '.')
        else:
            t = t.replace(',', '.')
        try:
            return float(t)
        except ValueError:
            return None
    return None


def flat(d, pre=''):
    """Вложенный JSON -> плоский словарь: {'a': {'b': 1}} -> {'a.b': 1}."""
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out.update(flat(v, f'{pre}{k}.'))
        else:
            out[f'{pre}{k}'] = v
    return out


def is_flag(k):
    k = k.split('.')[-1]
    return k.startswith(('is_', 'has_')) or k == 'stopcredit'


def is_measure_candidate(k):
    # служебные поля (идентификаторы, номера, названия) не считаются показателями
    last = k.split('.')[-1]
    return last not in NOT_MEASURES and not last.endswith('_id') and last not in ('name', 'code')


def parse(old, new, rule=''):
    """(номер договора, изменение, поле): изменение — 'рост', 'снижение', 'без изменения', 'установлен', 'снят'
    или '' (только факт события: старого значения нет или сравнивать нечего)."""
    od, nd = as_dict(old), as_dict(new)
    if od is None and nd is None:                           # значения не JSON, а просто числа / флаги
        a, b = as_num(old), as_num(new)
        if a is None or b is None:
            return None, '', ''
        if {a, b} <= {0.0, 1.0}:
            return None, ('установлен' if b > a else 'снят' if b < a else 'без изменения'), 'значение'
        return None, ('рост' if b > a else 'снижение' if b < a else 'без изменения'), 'значение'
    o, n = flat(od or {}), flat(nd or {})
    code = next((dd[k] for dd in (n, o) for k in dd if k.split('.')[-1] == 'contract_code' and dd[k] not in (None, '')),
                None)
    code = str(code).strip().upper() if code is not None else None
    for k in n:                                             # статусы: флаги is_* / has_* / stopcredit
        if is_flag(k) and k in o:
            a, b = as_num(o[k]), as_num(n[k])
            if a is not None and b is not None and a != b:
                return code, ('установлен' if b > a else 'снят'), k
    keys = [k for k in MEASURES if k in o and k in n] + \
           [k for k in n if k in o and k not in MEASURES and not is_flag(k) and is_measure_candidate(k)]
    vals = [(k, as_num(o[k]), as_num(n[k])) for k in keys]
    vals = [v for v in vals if v[1] is not None and v[2] is not None]
    if vals:                     # поле, о котором говорит название правила; иначе первое изменившееся числовое поле
        want = next((f for key, f in RULE_FIELD if key in rule.lower()), None)
        k, a, b = (next((v for v in vals if v[0].split('.')[-1] == want), None)
                   or next((v for v in vals if v[1] != v[2]), vals[0]))
        return code, ('рост' if b > a else 'снижение' if b < a else 'без изменения'), k
    return code, '', ''


parsed = [parse(a, b, r) for a, b, r in zip(ev['old'].values, ev['new'].values, ev['rule'].values)]
ev['code'] = [p[0] for p in parsed]
ev['change'] = [p[1] for p in parsed]
ev['field'] = [p[2] for p in parsed]

# проверка разбора на ваших данных: какие поля есть в JSON и как разобраны события каждого правила
def keys_of(vals):
    ks = {}
    for v in vals[:500]:
        d = as_dict(v)
        for k in (flat(d) if d else {}):
            if not k.split('.')[-1] in ('name',):
                ks[k] = ks.get(k, 0) + 1
    return ', '.join(sorted(ks, key=lambda k: -ks[k])[:8]) or '— (пусто)'


diag = []
for rule, g in ev.groupby('rule'):
    cnt = g['change'].replace('', 'только факт события').value_counts()
    fields = g.loc[g['field'] != '', 'field'].value_counts()
    diag.append({'Правило ПКБ (как в данных)': rule, 'Событий': num(len(g)),
                 'Поля в старом значении': keys_of(g['old'].values), 'Поля в новом значении': keys_of(g['new'].values),
                 'Как разобрано': '; '.join(f'{k} {num(v)}' for k, v in cnt.items()),
                 'По какому полю': ', '.join(fields.index[:3]) or '—'})
print('\nКак прочитаны старое и новое значение (JSON) по каждому правилу:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None, 'display.width', None):
    display(pd.DataFrame(diag))
ev = ev.drop(columns=['old', 'new'])

# 3) события по нашим собственным договорам не учитываются
own = set(norm_code(ovrd[C_CONTRACT]).dropna())
if C_PROCESS in ovrd.columns:
    own |= set(norm_code(ovrd[C_PROCESS]).dropna())
is_own = ev['code'].isin(own)
print(f'Из них по нашим договорам (не учитываются): {is_own.mean():.1%}')
ev = ev[~is_own]

# 4) показатель = правило + направление изменения (если правило его содержит)
ev['item'] = ev['rule'] + np.where(ev['change'] != '', ' — ' + ev['change'], '')


In [ ]:
# ---------------- РАСЧЁТ ПО КАЖДОМУ ПРАВИЛУ ----------------
# событие в окне [дата выхода − WINDOW_DAYS, дата выхода)
x = ev.merge(pop[['ep_id', 'iin', 'dlq_start']], on='iin')
x = x[(x['time'] < x['dlq_start']) & (x['time'] >= x['dlq_start'] - pd.Timedelta(days=WINDOW_DAYS))]
flags = x[['ep_id', 'item']].drop_duplicates()
y = pop['bad'].values
half = (pop['dlq_start'] > pop['dlq_start'].median()).values        # вторая половина периода
ev_dates = ev.groupby('item')['time'].agg(['min', 'max'])


def rr_ci(b1, n1, b0, n0):
    """Во сколько раз доля «плохих» с событием выше, чем без события; 95% ДИ и p-значение (лог-нормальное приближение)."""
    if min(n1, n0) == 0 or b0 == 0:
        return np.nan, np.nan, np.nan, np.nan
    c = 0.5 if b1 == 0 else 0.0                                     # поправка, если «плохих» с событием нет
    rr = ((b1 + c) / (n1 + c)) / (b0 / n0)
    se = math.sqrt(1 / (b1 + c) - 1 / (n1 + c) + 1 / b0 - 1 / n0)
    p = math.erfc(abs(math.log(rr)) / se / math.sqrt(2))
    return rr, rr * math.exp(-1.96 * se), rr * math.exp(1.96 * se), p


rows = []
for item, g in flags.groupby('item'):
    f = np.zeros(N, dtype=bool)
    f[g['ep_id'].values] = True
    n1, b1 = int(f.sum()), int(y[f].sum())
    rr, lo, hi, p = rr_ci(b1, n1, B - b1, N - n1)
    # IV флага (две группы: с событием / без)
    g1, b_1 = n1 - b1, b1
    g0, b_0 = (N - B) - g1, B - b1
    iv = sum((bb / B - gg / (N - B)) * math.log(((bb + 0.5) / B) / ((gg + 0.5) / (N - B)))
             for bb, gg in ((b_1, g1), (b_0, g0))) if 0 < B < N else np.nan
    halves = []
    for h in (False, True):                                         # риск по половинам периода
        hm = half == h
        fn, yn = f[hm], y[hm]
        halves.append(rr_ci(int(yn[fn].sum()), int(fn.sum()), int(yn[~fn].sum()), int((~fn).sum()))[0]
                      if fn.sum() >= MIN_EPISODES / 2 else np.nan)
    rule, _, change = item.partition(' — ')
    rows.append(dict(rule=rule, change=change, n1=n1, share=n1 / N, b1=b1, exp_b1=n1 * B / N,
                     br1=b1 / n1, br0=(B - b1) / (N - n1) if N > n1 else np.nan,
                     rr=rr, lo=lo, hi=hi, p=p, iv=iv, rr_h1=halves[0], rr_h2=halves[1],
                     first=ev_dates.at[item, 'min'], last=ev_dates.at[item, 'max']))
res = pd.DataFrame(rows)

# q-значение: p-значение с поправкой на число проверенных правил (Бенджамини–Хохберг)
ok = res['p'].notna()
pv = res.loc[ok, 'p'].values
order = np.argsort(pv)
q = np.empty_like(pv)
q[order] = np.minimum.accumulate((pv[order] * len(pv) / np.arange(1, len(pv) + 1))[::-1])[::-1].clip(max=1)
res['q'] = np.nan
res.loc[ok, 'q'] = q


def verdict(r):
    """Вывод по правилу: (порядок для сортировки, текст)."""
    up = r.rr > 1
    side = 'повышает риск' if up else 'снижает риск'
    one_side = r.lo > 1 or r.hi < 1
    if r.n1 < MIN_EPISODES or max(r.b1, r.exp_b1) < MIN_BADS:
        if one_side and r.q < 0.05:
            return 7, f'мало данных: предварительно {side}'
        return 8, 'мало данных — оценить нельзя'
    if not (r.q < 0.05):
        return (5, 'нет влияния') if (r.lo >= 1 / 1.15 and r.hi <= 1.15) else (6, 'влияние не доказано')
    far = r.lo if up else 1 / r.hi                                  # граница ДИ, ближняя к 1 (в «разах»)
    k = r.rr if up else 1 / r.rr
    if k >= 2 and far >= 1.4:
        return 1, f'сильное: {side}'
    if k >= 1.4 and far >= 1.15:
        return 2, f'среднее: {side}'
    if r.lo >= 1 / 1.15 and r.hi <= 1.15:
        return 4, 'практически нет влияния'
    return 3, f'слабое: {side}'


v = res.apply(verdict, axis=1)
res['order'], res['verdict'] = [a for a, _ in v], [b for _, b in v]
res['mag'] = np.log(res['rr']).abs()                                  # сила отличия от 1 в любую сторону
res = res.sort_values(['order', 'mag'], ascending=[True, False]).reset_index(drop=True)


In [ ]:
# ---------------- ИТОГ: ТЕКСТ, ТАБЛИЦА, ГРАФИК, EXCEL ----------------
def times(rr):
    if not np.isfinite(rr):
        return '—'
    if 0.995 <= rr <= 1.005:
        return 'так же'
    return f'в {rr:.2f} раза выше' if rr > 1 else f'в {1 / rr:.2f} раза ниже'


def times_ci(r):
    """Риск при событии с 95% ДИ в тех же словах («выше» / «ниже»)."""
    if r.b1 == 0:
        return f'«плохих» с событием нет (0 из {num(r.n1)})'
    if not np.isfinite(r.lo):
        return times(r.rr)
    if r.lo > 1:
        ci = f'в {r.lo:.2f}–{r.hi:.2f} раза выше'
    elif r.hi < 1:
        ci = f'в {1 / r.hi:.2f}–{1 / r.lo:.2f} раза ниже'
    else:
        ci = f'от {1 / r.lo:.2f} раза ниже до {r.hi:.2f} раза выше'
    return f'{times(r.rr)} (95% ДИ: {ci})'


def name(r):
    return r.rule + (f' — {r.change}' if r.change else '')


def group(prefix):
    sel = res[res['verdict'].str.startswith(prefix)]
    return [f'    • {name(r)}: ' + ('«плохих» с событием нет' if r.b1 == 0 else f'риск {times(r.rr)}') + f' (доля «плохих» {r.br1:.1%} против {r.br0:.1%}), событие у {r.share:.1%} эпизодов'
            for r in sel.itertuples()]


print(f'\nИТОГ (эпизоды с выходом в просрочку {A_TEXT}; {num(N)} эпизодов, «плохих» {B / N:.1%}; '
      + f'событие — за {WINDOW_DAYS} дн. до выхода в просрочку)')
for title, prefix in (('Сильно влияют', 'сильное'), ('Средне влияют', 'среднее'), ('Слабо влияют', 'слабое')):
    lines = group(prefix)
    print(f'\n{title}:' + ('' if lines else ' нет'))
    if lines:
        print('\n'.join(lines))
for title, pref in (('Практически не влияют / не влияют', ('практически', 'нет влияния')),
                    ('Влияние не доказано (разброс слишком большой)', ('влияние не доказано',)),
                    ('Мало данных — оценить нельзя', ('мало данных',))):
    sel = res[res['verdict'].str.startswith(pref)]
    print(f'\n{title}: ' + ('; '.join(name(r) + (f' ({r.verdict.split(": ")[1]})' if ': ' in r.verdict else '')
                                       for r in sel.itertuples()) or 'нет'))

out = pd.DataFrame({
    'Правило ПКБ (как в данных)': res['rule'],
    'Изменение': res['change'].replace('', '—'),
    'Вывод': res['verdict'],
    'Риск при событии': [times_ci(r) for r in res.itertuples()],
    'Эпизодов с событием': [f'{num(r.n1)} ({r.share:.1%})' for r in res.itertuples()],
    'Доля «плохих»: с событием / без': [f'{r.br1:.1%} / {r.br0:.1%}' for r in res.itertuples()],
    'IV': res['iv'].round(4),
    'Риск: 1-я / 2-я половина периода': [f'{a:.2f} / {b:.2f}'.replace('nan', '—') for a, b in zip(res['rr_h1'], res['rr_h2'])],
    'События правила в данных': [f'{fmt(a)}–{fmt(b)}' for a, b in zip(res['first'], res['last'])],
})
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None, 'display.width', None):
    display(out)

# график: во сколько раз меняется риск, с 95% ДИ (правила с «мало данных» не показаны)
gr = res[~res['verdict'].str.startswith('мало данных')].iloc[::-1]
if len(gr):
    lab = ['\n'.join(textwrap.wrap(name(r), 60)) for r in gr.itertuples()]
    col = ['#d62728' if 'повышает' in t else '#1f77b4' if 'снижает' in t else '#8c8c8c' for t in gr['verdict']]
    fig, ax = plt.subplots(figsize=(11, 0.55 * len(gr) + 1.5))
    yy = np.arange(len(gr))
    ax.hlines(yy, gr['lo'].clip(lower=0.1), gr['hi'].clip(upper=22), color=col, lw=2)
    ax.scatter(gr['rr'].clip(0.1, 22), yy, color=col, s=40, zorder=3)
    for yi, r in zip(yy, gr.itertuples()):
        ax.text(min(max(r.hi, r.rr), 22) * 1.03, yi, r.verdict, va='center', fontsize=8)
    ax.axvline(1, color='black', lw=0.8)
    ax.set_xscale('log')
    lo_x = max(min(gr['lo'].min(), gr['rr'].min(), 0.8) * 0.9, 0.1)
    hi_x = min(max(gr['hi'].max(), gr['rr'].max(), 1.25), 10) * 2.2
    ax.set_xlim(lo_x, hi_x)                                        # справа место для подписи вывода
    ticks = [t for t in (0.1, 0.25, 0.5, 0.67, 1, 1.5, 2, 3, 5, 10) if lo_x <= t <= hi_x]
    ax.xaxis.set_minor_locator(plt.NullLocator())
    ax.set_xticks(ticks)
    ax.set_xticklabels([f'{t:g}' for t in ticks])
    ax.set_yticks(yy)
    ax.set_yticklabels(lab, fontsize=8)
    ax.set_xlabel('Во сколько раз доля «плохих» с событием выше (>1) или ниже (<1), чем без события; линия — 95% ДИ')
    ax.set_title(f'Влияние правил ПКБ на переход в DPD {BAD_DPD}+ (событие за {WINDOW_DAYS} дн. до выхода в просрочку)\n'
                 'красный — повышает риск, синий — снижает, серый — нет влияния или не доказано', fontsize=10)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

# Excel: итог в одном листе
head = pd.DataFrame({'Правило ПКБ (как в данных)': [
    f'Эпизоды с выходом в просрочку: {A_TEXT}; данные ПКБ с {fmt(HES_START)} по {fmt(HES_END)}, '
    f'дни без данных: {gap_text(gap_days)}.',
    f'Эпизодов: {N}; «плохих» (DPD {BAD_DPD}+): {B / N:.1%}. Событие — за {WINDOW_DAYS} дн. до выхода в просрочку; '
    'события по нашим договорам не учитываются.', '']})
try:
    with pd.ExcelWriter(OUT_XLSX) as xw:
        head.to_excel(xw, sheet_name='Итог по правилам ПКБ', index=False, header=False)
        out.to_excel(xw, sheet_name='Итог по правилам ПКБ', index=False, startrow=len(head))
    print(f'\nСохранено: {OUT_XLSX}')
except Exception as e:                                              # нет openpyxl / xlsxwriter
    out.to_csv(OUT_XLSX.replace('.xlsx', '.csv'), index=False, encoding='utf-8-sig')
    print(f'\nExcel не записан ({e}); сохранено в CSV: {OUT_XLSX.replace(".xlsx", ".csv")}')


### Как читать вывод
* **сильное** — риск отличается в 2 раза и больше (и даже нижняя граница ДИ — в 1.4 раза и больше);
* **среднее** — в 1.4 раза и больше (граница ДИ — дальше 1.15);
* **слабое** — разница не случайна, но меньше;
* **практически нет влияния** — разница не случайна, но весь ДИ в пределах 0.87–1.15 раза;
* **нет влияния** — разница случайна, и весь ДИ в пределах 0.87–1.15 раза;
* **влияние не доказано** — разница может быть случайной, но ДИ слишком широкий, чтобы сказать «нет влияния»;
* **мало данных** — событие меньше чем у 200 эпизодов или меньше 30 «плохих» среди них. Если направление всё же видно, написано «предварительно».

«Случайна / не случайна» определяется с поправкой на число проверенных правил (q-значение < 0.05). **IV** — насколько правило разделяет «плохих» и «хороших» во всём портфеле, с учётом того, как часто бывает событие. **«Риск: 1-я / 2-я половина периода»** — тот же показатель отдельно по ранним и поздним эпизодам: если цифры сильно расходятся, эффект неустойчив.